In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Parameters
mu_m = 0.9       # h^-1
K_S = 0.21      # g/L
K_I = 1.5       # g/L
Y_XS = 0.6      # g/g
K_CL = 0.2      # mg/L
Y_OS = 160      # mg/g
M = 0.0314
V_L = 90        # L
G = 5000        # L/h
eps = 0.2
V_G = eps * V_L

C_Li = 6        # mg/L
C_Gi = 250      # mg/L
S_i = 30        # g/L

# Initial conditions
y0 = [1, 0, 6, 250]   # X, S, C_L, C_G

CL_limit = 3 * K_CL   # oxygen limitation threshold


def growth_rate(S, CL):
    return mu_m * S / (K_S + S + S**2 / K_I) * CL / (K_CL + CL)


def system(t, y, F, KLa):
    X, S, CL, CG = y
    
    mu = growth_rate(S, CL)
    CL_star = M * CG
    
    dXdt = -(F / V_L) * X + mu * X
    
    dSdt = (F / V_L) * (S_i - S) - (1 / Y_XS) * mu * X
    
    dCLdt = (F / V_L) * (C_Li - CL) - (Y_OS / Y_XS) * mu * X + KLa * (CL_star - CL)
    
    dCGdt = (G / V_G) * (C_Gi - CG) - KLa * (V_L / V_G) * (CL_star - CL)
    
    return [dXdt, dSdt, dCLdt, dCGdt]


def simulate(F, KLa, t_end=500):
    sol = solve_ivp(system, [0, t_end], y0, args=(F, KLa), method="BDF", rtol=1e-6, atol=1e-6)
    return sol


def is_not_oxygen_limited(F, KLa):
    sol = simulate(F, KLa)
    CL = sol.y[2]
    return np.min(CL) > CL_limit, np.min(CL), sol


F_values = np.linspace(6.5, 10, 8)
KLa_values = []

for F in F_values:
    for KLa in np.linspace(1, 300, 300):
        ok, CL_min, sol = is_not_oxygen_limited(F, KLa)
        if ok:
            KLa_values.append(KLa)
            print(f"F = {F:.2f} L/h, minimum KLa = {KLa:.1f} h^-1, min CL = {CL_min:.3f} mg/L")
            break